In [1]:
import plotly.graph_objects as go

def twh_to_ej(twh):
    """
    Convert Terawatt-hours (TWh) to Exajoules (EJ).

    Parameters:
    twh (float): Energy in Terawatt-hours.

    Returns:
    float: Energy in Exajoules.
    """
    conversion_factor = 0.0036
    return twh * conversion_factor

def xy(d):
    return zip(*sorted(d.items())) if d else ([], [])

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)

* Implemented Input Files
    * `/input/policy/korea-2035/power/gas_H2_blend_const_value.xml`
    * `/input/policy/korea-2035/power/gas_H2_blend_const_techs.xml`
    * `/input/policy/korea-2035/power/gas_const_value.xml`
    * `/input/policy/korea-2035/power/gas_const_techs.xml`
    * `/input/policy/korea-2035/power/gas_shutdown.xml`

# Gas

LNG generations (TWh) are projected for 2023, 2030, and 2035 in BPESD. The generation for 2025 is calculated by linear interpolation. These generations are prjected as the figure below.

In [2]:
dictCapTWh = {2020: 146.18, 2023: 157.7, 2030: 161, 2035: 101.1}

In [3]:
years_cap, values_cap = xy(dictCapTWh)

fig = go.Figure()

for name, x, y, dash in [
    ("Current Policies", years_cap, values_cap, None),
]:
    fig.add_trace(go.Scatter(
        x=list(x), y=list(y),
        mode='lines+markers',
        name=name,
        line=(dict(dash=dash) if dash else None)
    ))

# Build annotations without repeating blocks
target_years = [2020, 2023, 2030, 2035]
annotations = []
for yr in target_years:
    val = dictCapTWh.get(yr)
    if val is not None:
        annotations.append(go.layout.Annotation(
            x=yr, y=val,
            xanchor='center', yanchor='bottom',
            text=f"{val:.1f} TWh",
            showarrow=True, arrowhead=1, ax=0, ay=-20
        ))

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    width=800, height=600,
    annotations=annotations,
    xaxis=dict(
        title='Year',
        title_font=dict(size=18),
        tickfont=dict(size=15),
        tickvals=[2020, 2025, 2030, 2035],  # custom tick positions
        range=[2018, 2036]                  # xrange
    ),
    yaxis=dict(
        title='TWh',
        title_font=dict(size=18),
        tickfont=dict(size=15),
        range=[0, 200]                       # yrange
    ),
)

fig.write_image("../figures/gas_generation.png", scale=2)
fig.show()

However in the *Current policies* scenario, these projected generations include hydrogen co-firing generations. We calculated ammonia co-firing generation first and subtract this amount from total LNG generations to implement ceilings.

The 11th BPESD projects total hydrogen + ammonia power generation at 15.5 TWh in 2030, 32.8 TWh in 2035, and 43.9 TWh in 2038. However, it does not specify the individual generation amounts for hydrogen and ammonia. In contrast, the 10th BPESD projected hydrogen power generation of 6.1 TWh and ammonia power generation of 6.9 TWh in 2030.

In this study, it is assumed that the annual ratio of hydrogen to ammonia generation in the 11th Basic Plan is the same as the ratio in 2030 under the 10th Basic Plan. Accordingly, the hydrogen generation share is $\frac{6.1}{13} = 46.9\%$, and the ammonia generation share is $53.1\%$.

Annual conventional LNG generation is defined as LNG generation excluding hydrogen co-firing. In 2030 and 2035, the total coal–ammonia co-firing generation (50% hydrogen co-firing) amounts to:

$$15.5×0.469×2,32.8×0.469×2$$

In [4]:
15.5*0.469*2,32.8*0.469*2

(14.539, 30.766399999999994)

which correspond to 14.54 TWh and 30.77 TWh, respectively, for 2030 and 2035. Since only 50% of this generation is attributable to coal, we calculate:

$$14.54×0.5,30.77×0.5$$

In [5]:
14.54*0.5,30.77*0.5

(7.27, 15.385)

The resulting values (7.27 TWh and 15.385 TWh) are deducted from the 2030 and 2035 LNG generation figures to derive LNG generation net of co-firing. A 2025 value is obtained via linear interpolation to capture possible early hydrogen co-firing uptake.

In [6]:
print(f"year = 2025, hydrogen co-firing (TWh) = {twh_to_ej(14.54/7*2):.3f}")
print(f"year = 2030, hydrogen co-firing (TWh) = {twh_to_ej(14.54):.3f}")
print(f"year = 2035, hydrogen co-firing (TWh) = {twh_to_ej(30.77):.3f}")

year = 2025, hydrogen co-firing (TWh) = 0.015
year = 2030, hydrogen co-firing (TWh) = 0.052
year = 2035, hydrogen co-firing (TWh) = 0.111


As a by product we get ceilings for ammonia co-firing for *Current Policieis* scenario, `/input/policy/korea-2035/power/gas_H2_blend_const_value_cp.xml`

```xml

<?xml version="1.0" ?>
<scenario>
  <world>
    <region name="South Korea">
      <policy-portfolio-standard name="Gas-H2-Blend-Floor">
        <policyType>subsidy</policyType>
        <market>South Korea</market>
        <min-price year="2025">0</min-price>
        <min-price year="2030">0</min-price>
        <min-price year="2035">0</min-price>
        <constraint year="2025">0.014</constraint>
        <constraint year="2030">0.052</constraint>
        <constraint year="2035">0.111</constraint>
      </policy-portfolio-standard>
    </region>
  </world>
</scenario>


```

We now subtract co-firing generations from total LNG generations to implement ceilings for conventional coal powers: `/input/policy/korea-2035/power/gas_const_value.xml`

In [7]:
dictCapTWhRe = {2020: 146.18, 2023: 157.7, 2030: 161.0 - 7.27, 2035: 101.1  - 15.39}
dictCapTWhRe[2025] = dictCapTWhRe[2023] + (dictCapTWhRe[2030] - dictCapTWhRe[2023]) * (2/7)

In [8]:
print(f"year = 2020, gas (EJ) = {twh_to_ej(dictCapTWhRe[2020]):.3f}")
print(f"year = 2025, gas (EJ) = {twh_to_ej(dictCapTWhRe[2025]):.3f}")
print(f"year = 2030, gas (EJ) = {twh_to_ej(dictCapTWhRe[2030]):.3f}")
print(f"year = 2035, gas (EJ) = {twh_to_ej(dictCapTWhRe[2035]):.3f}")

year = 2020, gas (EJ) = 0.526
year = 2025, gas (EJ) = 0.564
year = 2030, gas (EJ) = 0.553
year = 2035, gas (EJ) = 0.309


```xml

<?xml version="1.0" ?>
<scenario>
  <world>
    <region name="South Korea">
      <policy-portfolio-standard name="Gas-Generation-Ceiling">
        <policyType>tax</policyType>
        <market>South Korea</market>
        <min-price year="2020">-10000</min-price>
        <min-price year="2025">-10000</min-price>
        <min-price year="2030">0</min-price>
        <min-price year="2035">0</min-price>
        <constraint year="2020">0.526</constraint>
        <constraint year="2025">0.563</constraint>
        <constraint year="2030">0.554</constraint>
        <constraint year="2035">0.309</constraint>
      </policy-portfolio-standard>
    </region>
  </world>
</scenario>

```

Additionally we assumed that 

* the capacity factors are adjusted to meet the generation ceiling: `/input/policy/korea-2035/power/gas_shutdown.xml`